# GPT-who-style experiments

This notebook evaluates GPT-who-style features from existing Stage 1 token-logprob Parquet rows. Configure `EXPERIMENTS` below to duplicate runs across scorer LLMs and training/evaluation datasets. It does not run inference or modify production classifier artifacts.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils.gpt_who_baseline_utils import (
    GPT_WHO_FEATURE_COLUMNS,
    build_gpt_who_feature_frame,
    fit_and_score_logistic,
    make_experiment_cohorts,
    summarize_scores,
)
from notebooks.utils.gpt_who_experiment_config import (
    DEFAULT_EXPERIMENTS,
    experiment_dataset_names,
    experiment_run_id,
    validate_experiments,
)
from notebooks.utils.logprob_metrics_distributions_utils import iter_logprob_row_batches

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
BASE_RUN_ID = os.environ.get("GPT_WHO_RUN_ID", "gpt-who-baseline-2026-09-26")
EXPERIMENTS = validate_experiments(DEFAULT_EXPERIMENTS)
WINDOW_SIZE = 20
SEED = 42
MIXED_MAX_PER_SPLIT = 500
TRAIN_COUNT_PER_CLASS = 900
EVAL_COUNT_PER_CLASS = 300
ROW_BATCH_SIZE = 256
OUTPUT_BASE = Path(os.environ.get("GPT_WHO_OUTPUT_ROOT", str(DATA_ROOT / "experiment-results" / "gpt-who-baseline")))


## Configure experiments

Edit `DEFAULT_EXPERIMENTS` in `notebooks/utils/gpt_who_experiment_config.py`, or replace `EXPERIMENTS` in the configuration cell with a validated tuple. Each experiment can select multiple scorer LLMs and separate training/evaluation dataset lists.

In [ ]:
all_metrics, all_scores, all_coefficients, all_exclusions, all_coverage = [], [], [], [], []

for experiment in EXPERIMENTS:
    experiment_name = experiment["name"]
    training_dataset_names = experiment_dataset_names(experiment, "training_datasets")
    evaluation_dataset_names = experiment_dataset_names(experiment, "evaluation_datasets")
    selected_datasets = tuple(dict.fromkeys((*training_dataset_names, *evaluation_dataset_names)))
    run_id = experiment_run_id(BASE_RUN_ID, experiment_name)

    for scorer_llm in experiment["scorer_llms"]:
        output_root = OUTPUT_BASE / f"{run_id}__{scorer_llm}"
        source_frames, feature_frames, exclusion_frames = [], [], []
        for rows in tqdm(iter_logprob_row_batches(
            PROCESSED_ROOT, selected_datasets=selected_datasets, selected_models=[scorer_llm], batch_size=ROW_BATCH_SIZE
        ), desc="Loading Stage 1 rows", unit="batch"):
            source_frames.append(pd.DataFrame([
                {"id": row.get("id"), "dataset": row.get("dataset"), "model": row.get("model", scorer_llm), "split": row.get("split")}
                for row in rows
            ]))
            batch_features, batch_exclusions = build_gpt_who_feature_frame(
                tqdm(rows, desc="Extracting GPT-who features", unit="row"), window_size=WINDOW_SIZE
            )
            feature_frames.append(batch_features)
            exclusion_frames.append(batch_exclusions)
        if not feature_frames:
            raise FileNotFoundError(
                f"No Stage 1 rows found for experiment={experiment_name!r}, scorer_llm={scorer_llm!r}, "
                f"datasets={selected_datasets!r} under {PROCESSED_ROOT}."
            )
        source_frame = pd.concat(source_frames, ignore_index=True)
        feature_frame = pd.concat(feature_frames, ignore_index=True)
        exclusions = pd.concat(exclusion_frames, ignore_index=True)
        if feature_frame.empty:
            raise ValueError(f"No usable rows remain for {experiment_name}/{scorer_llm}.")
        feature_frame["label"] = feature_frame["split"].map({"HG": 0, "MG": 1})
        unknown_splits = sorted(feature_frame.loc[feature_frame["label"].isna(), "split"].dropna().unique())
        if unknown_splits:
            raise ValueError(f"Unexpected split labels; expected HG/MG, got {unknown_splits}")
        feature_frame = feature_frame.dropna(subset=["id", "label"]).copy()

        cohorts = make_experiment_cohorts(
            feature_frame,
            training_datasets=training_dataset_names,
            evaluation_datasets=evaluation_dataset_names,
            seed=SEED,
            max_per_split=MIXED_MAX_PER_SPLIT,
            train_count_per_class=TRAIN_COUNT_PER_CLASS,
            eval_count_per_class=EVAL_COUNT_PER_CLASS,
        )
        train_keys = set(zip(cohorts["train"]["dataset"], cohorts["train"]["split"], cohorts["train"]["id"]))
        evaluation_keys = set(zip(cohorts["evaluation"]["dataset"], cohorts["evaluation"]["split"], cohorts["evaluation"]["id"]))
        if train_keys & evaluation_keys:
            raise ValueError(f"Parent IDs overlap for {experiment_name}/{scorer_llm}")
        if cohorts["train"]["label"].nunique() < 2 or cohorts["evaluation"]["label"].nunique() < 2:
            raise ValueError(f"Both cohorts must contain HG and MG rows for {experiment_name}/{scorer_llm}")

        scores, coefficients = fit_and_score_logistic(
            cohorts["train"], cohorts["evaluation"], feature_columns=GPT_WHO_FEATURE_COLUMNS, seed=SEED
        )
        metrics = summarize_scores(scores, partition="evaluation").assign(
            experiment_name=experiment_name, run_id=run_id, scorer_llm=scorer_llm,
            training_datasets=",".join(training_dataset_names), evaluation_datasets=",".join(evaluation_dataset_names),
        )
        scores = scores.assign(experiment_name=experiment_name, run_id=run_id, scorer_llm=scorer_llm)
        coefficients = coefficients.assign(experiment_name=experiment_name, run_id=run_id, scorer_llm=scorer_llm)
        exclusions = exclusions.assign(experiment_name=experiment_name, run_id=run_id, scorer_llm=scorer_llm)
        coverage = source_frame.groupby(["dataset", "model", "split"], dropna=False).size().rename("source_rows").reset_index()
        accepted = feature_frame.groupby(["dataset", "model", "split"], dropna=False).size().rename("accepted_rows").reset_index()
        excluded = exclusions.assign(excluded_rows=1).groupby(["dataset", "model", "split"], dropna=False)["excluded_rows"].sum().reset_index() if not exclusions.empty else pd.DataFrame(columns=["dataset", "model", "split", "excluded_rows"])
        coverage = coverage.merge(accepted, on=["dataset", "model", "split"], how="left").merge(excluded, on=["dataset", "model", "split"], how="left")
        coverage[["accepted_rows", "excluded_rows"]] = coverage[["accepted_rows", "excluded_rows"]].fillna(0).astype("int64")
        coverage = coverage.assign(experiment_name=experiment_name, run_id=run_id, scorer_llm=scorer_llm)

        output_root.mkdir(parents=True, exist_ok=True)
        metrics.to_parquet(output_root / "metrics.parquet", index=False)
        scores.to_parquet(output_root / "scores.parquet", index=False)
        coefficients.to_parquet(output_root / "coefficients.parquet", index=False)
        exclusions.to_parquet(output_root / "exclusions.parquet", index=False)
        coverage.to_parquet(output_root / "coverage.parquet", index=False)
        configuration = {
            "experiment_name": experiment_name, "run_id": run_id, "scorer_llm": scorer_llm,
            "training_datasets": list(training_dataset_names), "evaluation_datasets": list(evaluation_dataset_names),
            "window_size": WINDOW_SIZE, "max_per_split": MIXED_MAX_PER_SPLIT,
            "train_count_per_class": TRAIN_COUNT_PER_CLASS, "eval_count_per_class": EVAL_COUNT_PER_CLASS,
            "seed": SEED, "feature_columns": list(GPT_WHO_FEATURE_COLUMNS), "source_root": str(PROCESSED_ROOT),
            "output_root": str(output_root),
        }
        (output_root / "configuration.json").write_text(json.dumps(configuration, indent=2, sort_keys=True), encoding="utf-8")
        confusion = confusion_matrix(scores["label"], scores["predicted_label"], labels=[0, 1])
        fig_confusion, axis = plt.subplots(figsize=(4.5, 4.0), constrained_layout=True)
        ConfusionMatrixDisplay(confusion_matrix=confusion, display_labels=["HG", "MG"]).plot(ax=axis, cmap="Blues", colorbar=False)
        axis.set_title(f"GPT-who: {experiment_name}/{scorer_llm}")
        fig_scores, axis = plt.subplots(figsize=(7.0, 4.0), constrained_layout=True)
        sns.histplot(data=scores, x="mg_probability", hue="label", bins=20, stat="density", common_norm=False, element="step", palette={0: "#2563eb", 1: "#dc2626"}, ax=axis)
        axis.set_title(f"Held-out MG probability: {experiment_name}/{scorer_llm}")
        axis.set_xlabel("P(MG)")
        fig_confusion.savefig(output_root / "confusion_matrix.png", dpi=160, bbox_inches="tight")
        fig_scores.savefig(output_root / "score_distributions.png", dpi=160, bbox_inches="tight")
        plt.close(fig_confusion)
        plt.close(fig_scores)
        all_metrics.append(metrics)
        all_scores.append(scores)
        all_coefficients.append(coefficients)
        all_exclusions.append(exclusions)
        all_coverage.append(coverage)

metrics = pd.concat(all_metrics, ignore_index=True)
scores = pd.concat(all_scores, ignore_index=True)
coefficients = pd.concat(all_coefficients, ignore_index=True)
exclusions = pd.concat(all_exclusions, ignore_index=True)
coverage = pd.concat(all_coverage, ignore_index=True)
display(metrics)
display(coverage)

In [ ]:
display(pd.DataFrame({
    "experiment": metrics["experiment_name"],
    "scorer_llm": metrics["scorer_llm"],
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "roc_auc": metrics["roc_auc"],
}))